In [ ]:
import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt
import pywt

def pan_tompkins_preprocess(ecg_signal, fs=200):
    """
    Applies the classical Pan-Tompkins preprocessing pipeline:
    Cascaded Bandpass Filter -> Derivative -> Squaring -> Moving Window Integration (MWI)
    """
    # 1. Pan-Tompkins specific integer low-pass & high-pass filter (optimized for fs=200Hz)
    b_lp = np.zeros(13); b_lp[0], b_lp[6], b_lp[12] = 1, -2, 1
    a_lp = [1, -2, 1]
    lp_signal = signal.lfilter(b_lp, a_lp, ecg_signal)
    
    b_hp = np.zeros(33); b_hp[0], b_hp[16], b_hp[17], b_hp[32] = -1/32, 1, -1, 1/32
    a_hp = [1, -1]
    hp_signal = signal.lfilter(b_hp, a_hp, lp_signal)
    
    # 2. Derivative stage
    b_der = [2/8, 1/8, 0, -1/8, -2/8]
    der_signal = signal.lfilter(b_der, 1, hp_signal)
    
    # 3. Squaring stage
    sq_signal = der_signal ** 2
    
    # 4. Moving Window Integration (Window size approx 150ms)
    window_len = int(0.15 * fs)
    b_mwi = np.ones(window_len) / window_len
    mwi_signal = signal.lfilter(b_mwi, 1, sq_signal)
    
    return mwi_signal

def hilbert_envelope_preprocess(filtered_signal, fs=200):
    """
    Computes the magnitude of the analytic signal via Hilbert Transform,
    then applies a smoothing window to isolate the QRS envelope.
    """
    # Compute the analytical signal amplitude envelope
    analytic_signal = signal.hilbert(filtered_signal)
    amplitude_envelope = np.abs(analytic_signal)
    
    # Smooth the envelope using a moving average window
    window_len = int(0.15 * fs)
    b_mwi = np.ones(window_len) / window_len
    smoothed_envelope = signal.lfilter(b_mwi, 1, amplitude_envelope)
    
    return smoothed_envelope

def shannon_energy_preprocess(filtered_signal, fs=200):
    """
    Computes the Shannon Energy Envelope to emphasize the sharp spikes of the QRS complexes
    while suppressing lower amplitude T and P waves.
    """
    # Normalize the filtered signal intensity to a max of 1.0
    norm_signal = filtered_signal / np.max(np.abs(filtered_signal))
    
    # Avoid log(0) errors using a tiny epsilon value
    epsilon = 1e-10
    shannon_energy = - (norm_signal ** 2) * np.log(norm_signal ** 2 + epsilon)
    
    # Smooth the resulting energy wave
    window_len = int(0.15 * fs)
    b_mwi = np.ones(window_len) / window_len
    smoothed_shannon = signal.lfilter(b_mwi, 1, shannon_energy)
    
    return smoothed_shannon

def wavelet_transform_preprocess(ecg_signal, fs=200):
    """
    Applies Discrete Wavelet Transform (DWT) using PyWavelets.
    Decomposes signal using 'sym4' (Symlet 4), which matches QRS morphology well.
    Reconstructs using Detail coefficients where QRS energy resides (typically levels 3 & 4).
    """
    wavelet = 'sym4'
    level = 4
    
    # Perform Multilevel Discrete 1D Wavelet Decomposition
    coeffs = pywt.wavedec(ecg_signal, wavelet, level=level)
    
    # Isolate QRS complex energy: Clear Approximation (baseline) and lower details (high freq noise)
    # coeffs[0] = Approx, coeffs[1] = D1, coeffs[2] = D2, coeffs[3] = D3, coeffs[4] = D4
    for i in range(len(coeffs)):
        if i != 3 and i != 4:  # Keep only D3 and D4 coefficients
            coeffs[i] = np.zeros_like(coeffs[i])
            
    # Reconstruct signal using only the filtered coefficients
    reconstructed_signal = pywt.waverec(coeffs, wavelet)
    
    # Align length in case padding during waverec created a slight mismatch
    reconstructed_signal = reconstructed_signal[:len(ecg_signal)]
    
    # Return absolute magnitude as the final envelope
    return np.abs(reconstructed_signal)


# --- MAIN EXECUTION PIPELINE ---

path = 'G:\pulse_data\\relax\\06.txt'
fs = 500  # Target sampling rate (Hz)
s = 101

original_ecg = np.loadtxt(path, delimiter='\t', usecols=1)[s*2500:(s+1)*2500]
t = np.arange(len(original_ecg)) / fs

# 0. Pre-filtering stage (5-15 Hz Butterworth bandpass) for standard envelope routines
b_band, a_band = signal.butter(3, [5.0 / (0.5 * fs), 15.0 / (0.5 * fs)], btype='band')
bandpassed_ecg = signal.filtfilt(b_band, a_band, original_ecg)

# 1. Compute Preprocessing Methods
pt_output = pan_tompkins_preprocess(original_ecg, fs=fs)
hilbert_output = hilbert_envelope_preprocess(bandpassed_ecg, fs=fs)
shannon_output = shannon_energy_preprocess(bandpassed_ecg, fs=fs)
# wavelet_output = wavelet_transform_preprocess(original_ecg, fs=fs)

# 2. Visualizing and Comparing All Methods
fig, axes = plt.subplots(4, 1, figsize=(12, 11), sharex=True)

# Original Signal
axes[0].plot(t, original_ecg, color='black', lw=1.2, label='Raw ECG Signal')
axes[0].set_title('Comparison of ECG QRS-Complex Extraction Techniques', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend(loc='upper right')

# Pan-Tompkins
axes[1].plot(t, pt_output, color='crimson', lw=1.5, label='Pan-Tompkins (MWI)')
axes[1].set_ylabel('Amplitude')
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend(loc='upper right')

# Hilbert Envelope
axes[2].plot(t, hilbert_output, color='darkorange', lw=1.5, label='Hilbert Transform Envelope')
axes[2].set_ylabel('Amplitude')
axes[2].grid(True, linestyle='--', alpha=0.6)
axes[2].legend(loc='upper right')

# Shannon Energy Envelope
axes[3].plot(t, shannon_output, color='forestgreen', lw=1.5, label='Shannon Energy Envelope')
axes[3].set_ylabel('Amplitude')
axes[3].grid(True, linestyle='--', alpha=0.6)
axes[3].legend(loc='upper right')

# # Wavelet Transform
# axes[4].plot(t, wavelet_output, color='dodgerblue', lw=1.5, label='CWT (sym4, D3+D4 Envelope)')
# axes[4].set_xlabel('Time (seconds)', fontsize=11)
# axes[4].set_ylabel('Amplitude')
# axes[4].grid(True, linestyle='--', alpha=0.6)
# axes[4].legend(loc='upper right')

plt.tight_layout()
# plt.savefig('ecg_preprocessing_comparison.png', dpi=300)

In [ ]:
import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt
import pywt

def zero_phase_pan_tompkins(ecg_signal, fs=200):
    """
    Pan-Tompkins preprocessing using zero-phase Butterworth filtering
    to avoid both the Singular Matrix error and the peak delay.
    """
    # 1. Replace the unstable custom integer filters with a stable Butterworth bandpass
    lowcut = 5.0
    highcut = 15.0
    nyq = 0.5 * fs
    b_band, a_band = signal.butter(3, [lowcut / nyq, highcut / nyq], btype='band')
    
    # zero-phase forward-backward filtering
    bp_signal = signal.filtfilt(b_band, a_band, ecg_signal)
    
    # 2. Derivative stage
    b_der = [2/8, 1/8, 0, -1/8, -2/8]
    der_signal = signal.filtfilt(b_der, 1, bp_signal) # Zero phase derivative
    
    # 3. Squaring stage
    sq_signal = der_signal ** 2
    
    # 4. Moving Window Integration (MWI)
    # Using filtfilt here removes the 75ms lag introduced by the moving window!
    window_len = int(0.15 * fs)
    b_mwi = np.ones(window_len) / window_len
    mwi_signal = signal.filtfilt(b_mwi, 1, sq_signal) 
    
    return mwi_signal

def zero_phase_hilbert(filtered_signal, fs=200):
    analytic_signal = signal.hilbert(filtered_signal)
    amplitude_envelope = np.abs(analytic_signal)
    
    window_len = int(0.15 * fs)
    b_mwi = np.ones(window_len) / window_len
    return signal.filtfilt(b_mwi, 1, amplitude_envelope)

def zero_phase_shannon(filtered_signal, fs=200):
    norm_signal = filtered_signal / np.max(np.abs(filtered_signal))
    epsilon = 1e-10
    shannon_energy = - (norm_signal ** 2) * np.log(norm_signal ** 2 + epsilon)
    
    window_len = int(0.15 * fs)
    b_mwi = np.ones(window_len) / window_len
    return signal.filtfilt(b_mwi, 1, shannon_energy)

# --- EXECUTION ---
path = 'G:\pulse_data\\relax\\06.txt'
fs = 500  # Target sampling rate (Hz)
s = 101

original_ecg = np.loadtxt(path, delimiter='\t', usecols=1)[s*2500:(s+1)*2500]
t = np.arange(len(original_ecg)) / fs

# Global Zero-Phase Bandpass for Hilbert/Shannon
b_b, a_b = signal.butter(3, [5.0 / (0.5 * fs), 15.0 / (0.5 * fs)], btype='band')
bandpassed_ecg = signal.filtfilt(b_b, a_b, original_ecg)

# Processing with zero-phase methods
pt_output = zero_phase_pan_tompkins(original_ecg, fs=fs)
hilbert_output = zero_phase_hilbert(bandpassed_ecg, fs=fs)
shannon_output = zero_phase_shannon(bandpassed_ecg, fs=fs)

# Plotting to verify perfect alignment
fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)

axes[0].plot(t, original_ecg, color='black', label='Raw ECG')
axes[0].set_title('Zero-Phase Preprocessing (Peak Alignment)', fontsize=14, fontweight='bold')
axes[0].grid(True)
axes[0].legend()

axes[1].plot(t, pt_output, color='crimson', label='Pan-Tompkins (Zero-Phase)')
axes[1].grid(True)
axes[1].legend()

axes[2].plot(t, hilbert_output, color='darkorange', label='Hilbert Envelope (Zero-Phase)')
axes[2].grid(True)
axes[2].legend()

axes[3].plot(t, shannon_output, color='forestgreen', label='Shannon Energy (Zero-Phase)')
axes[3].grid(True)
axes[3].legend()

plt.tight_layout()
plt.show()